# 🕵️ Dia 31 — Perfis de fachada (sock puppets): quando são permitidos e como mantê-los limpos

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 31: Sock puppets: when they are allowed, and how to keep them clean* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-31.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Segunda, 12 de outubro. O `@serraazul.oficial` e o "Serra Azul Atendimento" mostram parte do conteúdo só para quem está logado. Você sugeriu criar uma conta "rapidinho". Rafa procurou "persona" no contrato e não achou.

*"Hoje a gente desenha no papel, decide por escrito, e não cria conta nenhuma."*

Neste notebook você vai:

1. **Codificar as três restrições** do original e rodar seis cenários.
2. **Caçar vínculos** no seu diagrama de compartimentos com busca em largura.
3. **Checar a lenda** contra as regras do original.
4. **Gerar o `persona-P1.md`** (planejamento; nenhuma conta).
5. **Jogar o quiz "descarta ou mantém?"**.

🐍 Só biblioteca padrão. Nenhuma rede necessária.

## 1. As três restrições, em código

Na ordem do fluxograma: política escrita → termos revisados e risco aceito → só passivo. A primeira resposta errada encerra.

In [ ]:
def decidir_persona(politica_escrita: bool, termos_e_risco_aceito: bool, so_passivo: bool):
    """Devolve (resultado, motivo) seguindo o fluxograma do Dia 31."""
    if not politica_escrita:                       # restrição 1: autorização
        return "PARE", "Sem política escrita que cubra a finalidade: planeje só no papel."
    if not termos_e_risco_aceito:                  # restrição 2: termos da plataforma
        return "PARE", "Termos não revisados ou ninguém aceitou o risco: aguarde esse aval."
    if not so_passivo:                             # restrição 3: uso passivo
        return "ESCALE", "Há interação (adicionar, grupo fechado, mensagem, suspeito): aval de advogado antes."
    return "CONSTRUA", "Construa com separação técnica completa e registre cada sessão."


CENARIOS = [
    # descrição, política?, termos+risco?, só passivo?
    ("Vigia, caso Isca: ver o @serraazul.oficial logado", False, False, True),
    ("Vigia, caso Isca: 'conversar com o atendimento falso para ver o roteiro'", False, False, False),
    ("Empresa com política, termos lidos, risco aceito pelo diretor; ver grupo ABERTO de denúncias", True, True, True),
    ("Mesma empresa; entrar num grupo FECHADO de vendedores do kit", True, True, False),
    ("Política existe, mas ninguém leu os termos do Instagram", True, False, True),
    ("Estudante, sozinho, 'só para treinar'", False, False, True),
]

for desc, pol, ter, pas in CENARIOS:
    resultado, motivo = decidir_persona(pol, ter, pas)
    print(f"• {desc}\n   -> {resultado}: {motivo}\n")

> 💡 Repare no cenário 2: ele para na restrição 1, mas **mesmo que** passasse pelas duas primeiras, cairia em ESCALE. Interagir com um suspeito não é visualização passiva, por mais inocente que pareça.

## 2. O que dá para fazer sem persona

Este é o resultado útil do PARE. Marque o que você já tem e veja o que falta.

In [ ]:
ALTERNATIVAS = [
    "Capturar o que o perfil mostra SEM login (nome, foto, bio, link, nº de seguidores), com hash e hora UTC",
    "Pedir à cooperativa as capturas que os cooperados já enviaram (com base legal e minimização)",
    "Registrar o número do 'Serra Azul Atendimento' a partir das mensagens recebidas pelos cooperados",
    "Seguir o link da bio até o domínio já conhecido (passivo, Dias 29-30)",
    "A cooperativa, como dona da marca, denunciar os perfis por falsa identidade às plataformas",
    "Preparar o material para a cooperativa pedir remoção/dados em juízo, se necessário",
]

MODO_JOGO = True   # False = só lista
feitos = 0
for i, alt in enumerate(ALTERNATIVAS, 1):
    print(f"[{i}] {alt}")
    if MODO_JOGO:
        r = input("    Já está no seu plano? (S/N): ").strip().upper()
        feitos += r.startswith("S")
if MODO_JOGO:
    print(f"\n{feitos}/{len(ALTERNATIVAS)} no plano. O Checkpoint BR pede pelo menos 4 itens na sua lista.")

## 3. 🔍 Caçando vínculos no diagrama (Passo 6)

Descreva o diagrama como **pares de células que se tocam**. Cada par tem um tipo:
- `"toca"` = ligação real e não mitigada (um vínculo!);
- `"amarelo"` = compartilhada, mas com mitigação escrita (bloqueia o caminho);
- linhas vermelhas ("nunca") simplesmente **não entram** na lista, porque não se tocam.

O código faz uma busca em largura a partir de cada célula da persona e procura qualquer caminho até uma célula `real:` que só passe por ligações `"toca"`.

In [ ]:
from collections import deque

# Exemplo: o diagrama do original, com DOIS erros plantados para você achar.
LIGACOES = [
    ("persona:vm", "real:notebook", "amarelo", "VM no mesmo notebook; mitigação: VM só para OSINT, snapshot limpo"),
    ("persona:navegador", "persona:vm", "toca", "o perfil roda dentro da VM"),
    ("persona:rede", "persona:vm", "toca", "a VM sai pela VPN"),
    ("persona:email", "persona:email2", "toca", "recuperação = 2ª caixa da persona"),
    # erro plantado 1: a 2ª caixa da persona tem como recuperação o celular pessoal
    ("persona:email2", "real:celular", "toca", "SMS de recuperação no celular pessoal (!)"),
    # erro plantado 2: o gerenciador de senhas sincroniza com o navegador pessoal
    ("persona:navegador", "real:gerenciador-senhas", "toca", "senhas da persona salvas no gerenciador pessoal (!)"),
    ("real:gerenciador-senhas", "real:navegador", "toca", "sincronizado com a conta pessoal"),
]


def cacar_vinculos(ligacoes):
    vizinhos = {}
    for a, b, tipo, nota in ligacoes:
        if tipo != "toca":                      # amarelo mitigado bloqueia; vermelho nem está aqui
            continue
        vizinhos.setdefault(a, []).append((b, nota))
        vizinhos.setdefault(b, []).append((a, nota))
    achados = []
    for inicio in sorted(n for n in vizinhos if n.startswith("persona:")):
        fila, visto = deque([(inicio, [inicio])]), {inicio}
        while fila:                                          # busca em largura: caminho mais curto primeiro
            no, caminho = fila.popleft()
            if no.startswith("real:"):
                achados.append(caminho)
                break
            for viz, _ in vizinhos.get(no, []):
                if viz not in visto:
                    visto.add(viz)
                    fila.append((viz, caminho + [viz]))
    return achados


vinculos = cacar_vinculos(LIGACOES)
if not vinculos:
    print("✅ Nenhum caminho até 'Eu de verdade' sem cruzar linha vermelha ou célula amarela.")
for c in vinculos:
    print("⚠️ VÍNCULO:", " → ".join(c))
print("\nCorrija cada vínculo (separe a célula) ou marque-o de amarelo COM mitigação escrita.")

In [ ]:
#@title Seu diagrama corrigido { display-mode: "form" }
#@markdown Ajuste as ligações problemáticas e rode de novo. Valor padrão = versão corrigida.
recuperacao_email2 = "nenhuma (recuperação só por códigos impressos guardados no cofre do caso)"  #@param {type:"string"}
senhas_da_persona = "cofre de senhas separado, dentro da VM, sem sincronização"  #@param {type:"string"}

LIGACOES_CORRIGIDAS = [l for l in LIGACOES if "(!)" not in l[3]]      # tira os dois erros
print("Recuperação da 2ª caixa:", recuperacao_email2)
print("Senhas da persona:", senhas_da_persona)
vinc = cacar_vinculos(LIGACOES_CORRIGIDAS)
print("✅ Sem vínculos." if not vinc else f"⚠️ Ainda há {len(vinc)} vínculo(s): {vinc}")

## 4. A lenda contra as regras do original

In [ ]:
#@title Lenda da persona P1 (planejamento) { display-mode: "form" }
nome = "nome genérico comum na região, conferido por busca"  #@param {type:"string"}
foto = "paisagem própria, sem rosto"  #@param ["nenhuma", "paisagem própria, sem rosto", "rosto gerado por IA", "foto de pessoa achada na internet"]
email_recuperacao = "segunda caixa só da persona"  #@param ["segunda caixa só da persona", "meu e-mail pessoal"]
telefone = "nenhum"  #@param ["nenhum", "número da organização para personas", "meu celular pessoal"]
cidade = "capital ampla da região"  #@param {type:"string"}

problemas = []
if foto == "foto de pessoa achada na internet":
    problemas.append("Foto roubada personifica alguém real e cai numa busca reversa de imagem.")
if foto == "rosto gerado por IA":
    problemas.append("Rosto de IA tem sinais conhecidos e pode ser tratado como inautêntico: prefira sem rosto.")
if email_recuperacao == "meu e-mail pessoal":
    problemas.append("Recuperação pelo e-mail pessoal liga a persona a você.")
if telefone == "meu celular pessoal":
    problemas.append("Telefone pessoal é o vínculo mais comum e o mais fácil de achar.")

print("✅ Lenda dentro das regras." if not problemas else "\n".join("⚠️ " + p for p in problemas))

## 5. 📝 O `persona-P1.md` (planejamento)

In [ ]:
from datetime import datetime, timezone

agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%MZ")
decisao, motivo = decidir_persona(False, False, True)        # o caso Isca, como está hoje

doc = f"""# persona-P1

Finalidade: ver conteúdo atrás de login dos perfis que imitam a Cooperativa Serra Azul (Operação Isca)
Autorizado por: só exercício de planejamento; nenhuma conta criada
Limite de atividade: só visualização passiva. Sem seguir, adicionar, entrar em grupos fechados, publicar, reagir ou mandar mensagens.

Decisão pelo fluxograma: {decisao} — {motivo}

## Lenda
| Campo | Valor | Observações |
|---|---|---|
| Nome | {nome} | |
| Foto | {foto} | nunca foto de pessoa real |
| E-mail | caixa nova só da persona | recuperação: {email_recuperacao} |
| Telefone | {telefone} | nunca o pessoal |
| Cidade | {cidade} | |
| Criada em | (não criada) | |

## Critérios de descarte
- Login a partir do meu IP real
- Plataforma sugere meus contatos reais
- Um alvo interage com a persona
- Pedido de verificação de identidade com documento
- Qualquer ação fora do limite de atividade no registro de uso

## Registro de uso
| Data/hora UTC | Plataforma | Finalidade | Caso | Ações realizadas |
|---|---|---|---|---|
| {agora} | nenhuma | sessão de planejamento | VIGIA-2026-001 | diagrama e lenda; nenhuma conta criada |

## Responsável pelo risco
Termos aplicáveis: Termos de Uso do Instagram; Termos do WhatsApp Business (seções de identidade).
Quem aceitou o risco: ninguém. A persona fica no papel.
"""
open("persona-P1.md", "w", encoding="utf-8").write(doc)
print(doc)
try:
    from google.colab import files
    files.download("persona-P1.md")
except ImportError:
    print("(Fora do Colab: persona-P1.md ficou na pasta atual.)")

## 6. 🎮 Quiz: descarta ou mantém?

Cada linha é um evento numa sessão de uma persona **já autorizada**. **D** = descarta a persona agora · **M** = mantém (registre e siga). `MODO_JOGO = False` mostra o gabarito.

In [ ]:
QUIZ = [
    ("Você abriu o perfil da persona no navegador pessoal 'só um segundo'.", "D", "Login fora do compartimento: vínculo criado."),
    ("A plataforma sugeriu seguir três colegas seus da Vigia.", "D", "Sugestão de contatos reais = a plataforma já ligou a persona a você."),
    ("O perfil observado publicou um story novo e você só viu.", "M", "Visualização passiva dentro do limite. Registre no uso."),
    ("O 'Serra Azul Atendimento' mandou mensagem para a persona.", "D", "Um alvo interagiu com a persona."),
    ("A plataforma pediu foto de documento para confirmar a identidade.", "D", "Critério de descarte do original."),
    ("A VPN caiu e a página recarregou pela sua conexão de casa.", "D", "Acesso a partir do IP real."),
    ("Você exportou a captura da página com hash e hora UTC.", "M", "Coleta normal; vai para o registro de uso."),
    ("Você curtiu uma publicação 'para a conta parecer real'.", "D", "Ação fora do limite de atividade: é a deriva da Camada 8."),
]

MODO_JOGO = True
acertos = 0
for i, (evento, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {evento}")
    if MODO_JOGO:
        r = ""
        while r not in ("D", "M"):
            r = input("   Descarta ou mantém? (D/M): ").strip().upper() or "?"
            if r not in ("D", "M"):
                r = {"S": "D", "N": "M"}.get(r, r)          # aceita S/N como atalho
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end=" ")
    print(f"Gabarito: {certo}. {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## ✅ Antes de fechar o Dia 31

- [ ] `notas/decisao-persona-isca.md` termina em PARE, com a fonte de cada resposta
- [ ] Lista "sem persona" com pelo menos 4 itens
- [ ] Verificador de vínculos sem nenhum caminho aberto
- [ ] `persona-P1.md` gerado; responsável pelo risco = "ninguém" → nenhuma conta

**Amanhã (Dia 32):** o que o servidor do IscaKit vê de quem o visita. Rafa vai mostrar um log ilustrativo, e uma das linhas é da própria Vigia.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>